In [1]:
import numpy as np

def initialize(context):
    # ========== 策略参数 ==========
    g.window = 60               # PME因子计算窗口
    g.N = 20                    # 持仓数量
    g.rebalance_days = 10       # 调仓频率（天）
    g.stop_loss = -0.15         # 止损线（-15%）
    g.last_trade_date = None
    
    # 融合权重（等权）
    g.weight_pme = 1.0          # PME因子权重
    g.weight_roe = 1.0          # ROE因子权重
    g.weight_cap = 1.0          # 市值因子权重
    
    # ========== 股票池选择 ==========
    # 选项1：沪深300（大盘股）
    # g.stock_pool = get_index_stocks('000300.XSHG')
    
    # 选项2：中证500（中小盘，推荐！）
    g.stock_pool = get_index_stocks('000905.XSHG')
    
    # 选项3：自定义（测试用）
    # g.stock_pool = get_index_stocks('000300.XSHG')[:100]
    
    log.info(f'股票池大小: {len(g.stock_pool)}')
    log.info(f'股票池类型: {"中证500" if "000905" in str(g.stock_pool) else "沪深300"}')
    
    # ========== 可选：行业分散 ==========
    g.use_sector_diversification = False   # 默认关闭，可改为True
    g.max_stocks_per_sector = 5             # 每个行业最多持仓数
    
    # ========== 手续费和滑点 ==========
    set_commission(PerTrade(buy_cost=0.0003, sell_cost=0.0013, min_cost=5))
    set_slippage(FixedSlippage(0.002))
    set_benchmark('000300.XSHG')
    
    # 每天收盘后运行
    run_daily(trade, time='close')
    
    log.info('=' * 70)
    log.info('优化版三因子融合策略：PME + 市值 + ROE')
    log.info(f'持仓数量: {g.N}')
    log.info(f'调仓频率: {g.rebalance_days}天')
    log.info(f'止损线: {g.stop_loss:.0%}')
    log.info(f'行业分散: {"开启" if g.use_sector_diversification else "关闭"}')
    log.info('=' * 70)


def trade(context):
    current_date = context.current_dt.strftime('%Y-%m-%d')
    
    # ========== 0. 止损检查（每天执行） ==========
    stop_loss_sell(context)
    
    # 控制调仓频率
    if g.last_trade_date is not None:
        days_diff = (context.current_dt.date() - g.last_trade_date).days
        if days_diff < g.rebalance_days:
            return
    
    log.info(f'\n========== 调仓日: {current_date} ==========')
    
    # ========== 1. 获取有效股票 ==========
    valid_stocks = []
    for stock in g.stock_pool[:200]:  # 限制计算量
        try:
            hist = attribute_history(stock, g.window + 30, '1d', ['close', 'volume'], skip_paused=True)
            if hist is not None and len(hist) >= g.window:
                if hist['close'].iloc[-1] > 0:
                    valid_stocks.append(stock)
        except:
            continue
    
    if len(valid_stocks) < g.N:
        log.warn(f'有效股票不足: {len(valid_stocks)}')
        return
    
    # ========== 2. 计算三个因子 ==========
    pme_scores = {}      # PME因子（越高越好）
    roe_scores = {}      # ROE因子（越高越好）
    cap_scores = {}      # 市值因子（越小越好）
    
    for stock in valid_stocks:
        try:
            # 2.1 PME因子
            hist = attribute_history(stock, g.window + 30, '1d', ['close', 'volume'], skip_paused=True)
            if hist is None or len(hist) < g.window:
                continue
            
            close = hist['close'].values
            volume = hist['volume'].values
            m = calculate_pme_m(close, volume, g.window)
            
            if m is not None and not np.isnan(m):
                # 反转m因子（根据之前测试，反转后效果更好）
                m = 2.0 - m
                m = np.clip(m, 0.3, 1.7)
                pme_scores[stock] = m
            
            # 2.2 ROE因子
            roe_data = get_fundamentals(
                query(valuation.code, indicator.roe).filter(valuation.code == stock)
            )
            if not roe_data.empty:
                roe = roe_data['roe'].iloc[0]
                if not np.isnan(roe):
                    roe_scores[stock] = roe
            
            # 2.3 市值因子
            cap_data = get_fundamentals(
                query(valuation.code, valuation.market_cap).filter(valuation.code == stock)
            )
            if not cap_data.empty:
                cap = cap_data['market_cap'].iloc[0]
                if cap > 0:
                    cap_scores[stock] = 1.0 / cap  # 倒数，小市值得分高
                    
        except Exception as e:
            continue
    
    # 确保有足够数据
    common_stocks = set(pme_scores.keys()) & set(roe_scores.keys()) & set(cap_scores.keys())
    if len(common_stocks) < g.N:
        log.warn(f'共同有效股票不足: {len(common_stocks)}')
        return
    
    # ========== 3. 因子标准化（排名法） ==========
    def rank_normalize(scores_dict):
        if len(scores_dict) == 0:
            return {}
        items = list(scores_dict.items())
        items.sort(key=lambda x: x[1])
        ranks = {}
        for i, (stock, _) in enumerate(items):
            ranks[stock] = i + 1
        return ranks
    
    pme_rank = rank_normalize(pme_scores)
    roe_rank = rank_normalize(roe_scores)
    cap_rank = rank_normalize(cap_scores)
    
    # ========== 4. 综合得分 ==========
    total_scores = {}
    for stock in common_stocks:
        total_scores[stock] = (
            g.weight_pme * pme_rank.get(stock, 0) +
            g.weight_roe * roe_rank.get(stock, 0) +
            g.weight_cap * cap_rank.get(stock, 0)
        )
    
    # ========== 5. 选股 ==========
    sorted_stocks = sorted(total_scores.items(), key=lambda x: x[1], reverse=True)
    buy_list = [s[0] for s in sorted_stocks[:g.N]]
    
    # 可选：行业分散
    if g.use_sector_diversification:
        buy_list = sector_diversification(buy_list, g.max_stocks_per_sector)
    
    # 打印Top10
    log.info('Top10 综合得分:')
    for stock, score in sorted_stocks[:10]:
        log.info(f'  {stock}: 总分={score:.0f} '
                 f'(PME={pme_rank.get(stock, 0):.0f}, '
                 f'ROE={roe_rank.get(stock, 0):.0f}, '
                 f'市值={cap_rank.get(stock, 0):.0f})')
    
    # 打印PME统计
    m_values = list(pme_scores.values())
    log.info(f'PME统计: 均值={np.mean(m_values):.4f}, m>1比例={sum(1 for m in m_values if m>1)/len(m_values):.1%}')
    
    # ========== 6. 执行调仓 ==========
    # 卖出
    for stock in list(context.portfolio.positions.keys()):
        if stock not in buy_list:
            order_target(stock, 0)
            log.info(f'卖出: {stock}')
    
    # 买入
    if len(buy_list) > 0:
        target_value = context.portfolio.total_value / len(buy_list)
        for stock in buy_list:
            order_target_value(stock, target_value)
    
    log.info(f'调仓完成，目标持仓: {len(buy_list)}只')
    log.info(f'实际持仓: {len(context.portfolio.positions)}只')
    log.info(f'账户总资产: {context.portfolio.total_value:.2f}')
    
    g.last_trade_date = context.current_dt.date()


def calculate_pme_m(close, volume, window):
    """计算PME扩散指数m"""
    try:
        returns = np.abs(np.diff(close) / (close[:-1] + 1e-8))
        amount = close * volume
        amount_ma = np.convolve(amount, np.ones(window)/window, mode='valid')
        
        if len(amount_ma) < 20:
            return None
        
        min_len = min(len(returns), len(amount_ma))
        if min_len < 20:
            return None
        
        returns = returns[-min_len:]
        amount_short = amount[-min_len:]
        amount_ma = amount_ma[-min_len:]
        
        u = returns * (amount_short / (amount_ma + 1e-8))
        u = u[~np.isnan(u)]
        u = u[~np.isinf(u)]
        
        if len(u) < 15:
            return None
        
        u_log = np.log(u + 1e-6)
        du = np.diff(u_log)
        
        min_len_corr = min(len(u_log) - 1, len(du))
        if min_len_corr < 5:
            return 0.8
        
        corr = np.corrcoef(u_log[:min_len_corr], du[:min_len_corr])[0, 1]
        
        if np.isnan(corr):
            return 1.0
        
        m = 1.0 + corr
        return max(0.3, min(1.7, m))
        
    except Exception as e:
        return None


def stop_loss_sell(context):
    """止损：单只股票亏损超过阈值则卖出"""
    for stock in list(context.portfolio.positions.keys()):
        position = context.portfolio.positions[stock]
        if position.total_amount > 0:
            # 计算盈亏比例
            pnl_ratio = (position.price - position.avg_cost) / position.avg_cost
            if pnl_ratio < g.stop_loss:
                order_target(stock, 0)
                log.info(f'止损卖出: {stock}, 亏损: {pnl_ratio:.2%}')


def sector_diversification(buy_list, max_per_sector):
    """行业分散：确保每个行业不超过指定数量"""
    try:
        # 获取股票行业分类
        sector_dict = {}
        for stock in buy_list:
            # 获取行业代码（这里简化，实际可用get_industry）
            sector_dict[stock] = 'unknown'
        
        # 按行业分组
        sector_groups = {}
        for stock, sector in sector_dict.items():
            if sector not in sector_groups:
                sector_groups[sector] = []
            sector_groups[sector].append(stock)
        
        # 每个行业最多取 max_per_sector 只
        diversified = []
        for sector, stocks in sector_groups.items():
            diversified.extend(stocks[:max_per_sector])
        
        return diversified[:len(buy_list)]
    except:
        return buy_list


def after_trading_end(context):
    pass

/opt/pyenv/versions/3.11.8/lib/python3.11/site-packages/bigmodule/requires.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.


[2026-01-08 10:51:33] [info     ] cn_stock_basic_selector.v8 开始运行 ..
[2026-01-08 10:51:34] [info     ] cn_stock_basic_selector.v8 命中缓存
[2026-01-08 10:51:34] [info     ] cn_stock_basic_selector.v8 运行完成 [0.516s].
[2026-01-08 10:51:34] [info     ] input_features_dai.v30 开始运行 ..
[2026-01-08 10:51:34] [info     ] input_features_dai.v30 命中缓存
[2026-01-08 10:51:34] [info     ] input_features_dai.v30 运行完成 [0.106s].
[2026-01-08 10:51:34] [info     ] score_to_position.v5 开始运行 ..
[2026-01-08 10:51:34] [info     ] score_to_position.v5 命中缓存
[2026-01-08 10:51:34] [info     ] score_to_position.v5 运行完成 [0.031s].
[2026-01-08 10:51:34] [info     ] extract_data_dai.v20 开始运行 ..
[2026-01-08 10:51:34] [info     ] extract_data_dai.v20 命中缓存
[2026-01-08 10:51:34] [info     ] extract_data_dai.v20 运行完成 [0.035s].
[2026-01-08 10:51:35] [info     ] bigtrader.v53 开始运行 ..
[2026-01-08 10:51:35] [info     ] got metadata extra from input datasource
[2026-01-08 10:51:35] [info     ] read input 'data' ..
[2026-01-08 10:51:

[2026-01-08 10:51:41] [info     ] bigtrader.v53 运行完成 [6.378s].
